## 2. 卷积和池化层

### 2.1 理论计算题

#### 1. 输出特征图（Feature Map）尺寸计算

**已知条件：**
*   输入高和宽：$H_{in} = 32, W_{in} = 32$
*   卷积核大小：$k_h = 5, k_w = 5$
*   填充：$p = 2$
*   步幅：$s = 2$
*   输出通道数（即卷积核数量）：$C_{out} = 16$

**特征图高和宽计算公式：**
$$ H_{out} = \lfloor \frac{H_{in} - k_h + 2p}{s} \rfloor + 1 $$
$$ W_{out} = \lfloor \frac{W_{in} - k_w + 2p}{s} \rfloor + 1 $$

代入数值计算：
$$ H_{out} = \lfloor \frac{32 - 5 + 2 \times 2}{2} \rfloor + 1 = \lfloor \frac{31}{2} \rfloor + 1 = 15 + 1 = 16 $$
$$ W_{out} = \lfloor \frac{32 - 5 + 2 \times 2}{2} \rfloor + 1 = 16 $$

**答：** 输出的特征图尺寸为 **$16 \times 16 \times 16$** （通道数 $\times$ 高 $\times$ 宽）。

---

#### 2. 单个输出像素点的点乘操作次数计算

**分析：**
为了计算输出特征图中单个通道内的某一个像素值，我们需要将一个大小为 $C_{in} \times k_h \times k_w$ 的三维卷积核与输入对应局部区域进行元素相乘并求和（点乘）。

*   输入通道数：$C_{in} = 3$
*   卷积核高与宽：$k_h = 5, k_w = 5$

单个像素点的乘法次数为：
$$ \text{Multiplications} = C_{in} \times k_h \times k_w = 3 \times 5 \times 5 = 75 $$

**答：** 计算单个输出通道的一个像素值，需要对输入进行 **$75$** 次点乘（乘法）操作。

### 2.2 编程题：二维最大池化（Max Pooling）从零实现

In [2]:
import numpy as np
import torch

def max_pool2d(X, pool_size, stride, padding):
    """
    手动实现支持步幅和填充的 2D 最大池化前向传播
    支持输入 X 形状为 (batch_size, channels, height, width) 的 PyTorch 张量或 NumPy 数组
    """
    is_torch = isinstance(X, torch.Tensor)
    if is_torch:
        X_np = X.detach().cpu().numpy()
    else:
        X_np = np.array(X)
        
    N, C, H, W = X_np.shape
    kh, kw = pool_size if isinstance(pool_size, tuple) else (pool_size, pool_size)
    sh, sw = stride if isinstance(stride, tuple) else (stride, stride)
    ph, pw = padding if isinstance(padding, tuple) else (padding, padding)
    
    # 用 -inf 填充，防止输入中存在负值时，边缘补 0 干扰最大值计算
    X_padded = np.pad(X_np, ((0,0), (0,0), (ph, ph), (pw, pw)), 
                      mode='constant', constant_values=-np.inf)
    
    out_h = (H + 2 * ph - kh) // sh + 1
    out_w = (W + 2 * pw - kw) // sw + 1
    
    Y = np.zeros((N, C, out_h, out_w))
    
    for i in range(out_h):
        for j in range(out_w):
            h_start = i * sh
            h_end = h_start + kh
            w_start = j * sw
            w_end = w_start + kw
            # 取窗口内的最大值
            Y[:, :, i, j] = np.max(X_padded[:, :, h_start:h_end, w_start:w_end], axis=(2, 3))
            
    if is_torch:
        return torch.from_numpy(Y).to(X.device)
    return Y

# 验证测试
X_test = torch.arange(16, dtype=torch.float32).reshape(1, 1, 4, 4)
print("输入数据:\n", X_test)
Y_test = max_pool2d(X_test, pool_size=2, stride=2, padding=0)
print("池化输出:\n", Y_test)

输入数据:
 tensor([[[[ 0.,  1.,  2.,  3.],
          [ 4.,  5.,  6.,  7.],
          [ 8.,  9., 10., 11.],
          [12., 13., 14., 15.]]]])
池化输出:
 tensor([[[[ 5.,  7.],
          [13., 15.]]]], dtype=torch.float64)


## 3. LeNet, AlexNet, VGG 和 NiN

### 3.1 理论计算题

假设输入和输出的特征图通道数均为 $C$，且无偏置。

#### 1. 计算一个 $5 \times 5$ 卷积层的参数量
一个卷积核的参数量为：$5 \times 5 \times C$。
由于输出通道数也是 $C$，所以共有 $C$ 个卷积核。
$$ \text{Parameters} = 5 \times 5 \times C \times C = 25C^2 $$

---

#### 2. 计算两个串联的 $3 \times 3$ 卷积层的总参数量
*   第一层（输入通道数 $C$，输出通道数 $C$）：参数量为 $3 \times 3 \times C \times C = 9C^2$
*   第二层（输入通道数 $C$，输出通道数 $C$）：参数量为 $3 \times 3 \times C \times C = 9C^2$
$$ \text{Total Parameters} = 9C^2 + 9C^2 = 18C^2 $$

**分析结论：**
可以看出，两个串联的 $3 \times 3$ 卷积层具有与一个 $5 \times 5$ 卷积层相同的感受野（Receptive Field），但参数量却减少了（从 $25C^2$ 降至 $18C^2$），且能提供更多的非线性激活，这就是 VGG 频繁使用小卷积核级联的核心原因。

### 3.2 编程题：定义一个 NiN 块

In [3]:
import torch.nn as nn

def nin_block(in_channels, out_channels, kernel_size, stride, padding):
    """
    定义一个标准的 NiN 块 (由1个普通卷积和2个1x1卷积级联而成)
    """
    return nn.Sequential(
        # 1. 普通卷积层
        nn.Conv2d(in_channels, out_channels, kernel_size, stride, padding),
        nn.ReLU(),
        # 2. 第一个 1x1 卷积层
        nn.Conv2d(out_channels, out_channels, kernel_size=1),
        nn.ReLU(),
        # 3. 第二个 1x1 卷积层
        nn.Conv2d(out_channels, out_channels, kernel_size=1),
        nn.ReLU()
    )

# 测试 NiN 块结构
test_block = nin_block(in_channels=3, out_channels=64, kernel_size=3, stride=1, padding=1)
print(test_block)

Sequential(
  (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (1): ReLU()
  (2): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1))
  (3): ReLU()
  (4): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1))
  (5): ReLU()
)


## 4. Inception, 批量归一化和残差网络

### 4.1 理论计算题

**已知数据：**
样本数据：$x_1 = 2, x_2 = 4, x_3 = 6, x_4 = 8$
缩放参数 $\gamma = 2$，平移参数 $\beta = 1$，常数 $\epsilon = 0$。

#### 1. 计算均值 $\mu$ 与方差 $\sigma^2$
$$ \mu = \frac{2 + 4 + 6 + 8}{4} = \frac{20}{4} = 5 $$
$$ \sigma^2 = \frac{(2-5)^2 + (4-5)^2 + (6-5)^2 + (8-5)^2}{4} = \frac{9 + 1 + 1 + 9}{4} = \frac{20}{4} = 5 $$
标准差：$\sigma = \sqrt{5}$

#### 2. 计算归一化值 $\hat{x}_i$
由于 $\epsilon = 0$，归一化公式为 $\hat{x}_i = \frac{x_i - \mu}{\sqrt{\sigma^2}}$：
*   $\hat{x}_1 = \frac{2 - 5}{\sqrt{5}} = -\frac{3}{\sqrt{5}}$
*   $\hat{x}_2 = \frac{4 - 5}{\sqrt{5}} = -\frac{1}{\sqrt{5}}$
*   $\hat{x}_3 = \frac{6 - 5}{\sqrt{5}} = \frac{1}{\sqrt{5}}$
*   $\hat{x}_4 = \frac{8 - 5}{\sqrt{5}} = \frac{3}{\sqrt{5}}$

#### 3. 计算最终变换后的输出 $y_i = \gamma \hat{x}_i + \beta = 2 \hat{x}_i + 1$
*   $y_1 = 2 \times \left(-\frac{3}{\sqrt{5}}\right) + 1 = 1 - \frac{6}{\sqrt{5}} \approx 1 - 2.683 = -1.683$
*   $y_2 = 2 \times \left(-\frac{1}{\sqrt{5}}\right) + 1 = 1 - \frac{2}{\sqrt{5}} \approx 1 - 0.894 = 0.106$
*   $y_3 = 2 \times \left(\frac{1}{\sqrt{5}}\right) + 1 = 1 + \frac{2}{\sqrt{5}} \approx 1 + 0.894 = 1.894$
*   $y_4 = 2 \times \left(\frac{3}{\sqrt{5}}\right) + 1 = 1 + \frac{6}{\sqrt{5}} \approx 1 + 2.683 = 3.683$

**答：** 最终输出值为：
*   $y_1 \approx -1.683$
*   $y_2 \approx 0.106$
*   $y_3 \approx 1.894$
*   $y_4 \approx 3.683$

### 4.2 编程题：自定义 ResNet 残差块

In [4]:
import torch
import torch.nn as nn
from torch.nn import functional as F

class Residual(nn.Module):
    """
    自定义 ResNet 残差块类
    """
    def __init__(self, input_channels, num_channels, use_1x1conv=False, strides=1):
        super().__init__()
        self.conv1 = nn.Conv2d(input_channels, num_channels, kernel_size=3, padding=1, stride=strides)
        self.conv2 = nn.Conv2d(num_channels, num_channels, kernel_size=3, padding=1)
        
        if use_1x1conv:
            self.conv3 = nn.Conv2d(input_channels, num_channels, kernel_size=1, stride=strides)
        else:
            self.conv3 = None
            
        self.bn1 = nn.BatchNorm2d(num_channels)
        self.bn2 = nn.BatchNorm2d(num_channels)

    def forward(self, X):
        Y = F.relu(self.bn1(self.conv1(X)))
        Y = self.bn2(self.conv2(Y))
        if self.conv3:
            X = self.conv3(X)
        Y += X  # 残差连接 (f(x) + x)
        return F.relu(Y)

# 测试残差块前向传播
res_block_no_1x1 = Residual(input_channels=3, num_channels=3)
res_block_with_1x1 = Residual(input_channels=3, num_channels=6, use_1x1conv=True, strides=2)

X = torch.randn(4, 3, 64, 64)
print("不带 1x1 卷积输出:", res_block_no_1x1(X).shape)
print("带 1x1 卷积（降采样）输出:", res_block_with_1x1(X).shape)

不带 1x1 卷积输出: torch.Size([4, 3, 64, 64])
带 1x1 卷积（降采样）输出: torch.Size([4, 6, 32, 32])


## 5. 图像增广，微调和样式迁移

### 5.1 理论计算题

#### 1. 特征提取层设置较小学习率，输出层设置较大学习率的原因：
*   **底层特征提取层** 在大型源数据集（如 ImageNet）上进行了训练，已经学到了极强的通用特征抽取能力（如提取边缘、纹理、色块、简单形状等）。这些特征在新的目标数据集上同样适用。因此，设置很小的学习率（甚至冻结）可以**防止破坏已经学到的优秀通用特征，并能极大加速收敛，避免发生过拟合**。
*   **新初始化的顶层输出层** 则完全是特定于新任务的目标分类。因为这部分权重是随机初始化的，它需要快速调整以适应新目标数据集的类别定义，所以需要使用**较大的学习率**来进行充分训练。

---

#### 2. 如果目标数据集非常小，且与源数据集非常相似，应采取什么微调策略？
*   **策略：** 我们应该采取**直接冻结模型所有的底层特征提取层**，仅仅微调和训练最终的**输出分类层（线性层）**的策略。
*   **原因：** 因为数据集高度相似，源模型抽取的特征已经天然完全适用于目标数据集；又由于目标数据集规模极小，哪怕训练很少的卷积层也极易引起严重的过拟合，因此只训练最后一层分类器是最稳妥、效果最好的方案。

### 5.2 编程题：图像增广管道创建

In [5]:
from torchvision import transforms

# 创建组合图像增广管道
train_augs = transforms.Compose([
    # 1. 随机裁剪，面积比例 0.08 到 1.0，并缩放到 224x224
    transforms.RandomResizedCrop(224, scale=(0.08, 1.0)),
    # 2. 50% 概率水平翻转
    transforms.RandomHorizontalFlip(p=0.5),
    # 3. 随机改变亮度、对比度、饱和度，范围设为 0.5
    transforms.ColorJitter(brightness=0.5, contrast=0.5, saturation=0.5),
    # 4. 转换为 Tensor 张量
    transforms.ToTensor()
])

print("增广管道创建成功:\n", train_augs)

增广管道创建成功:
 Compose(
    RandomResizedCrop(size=(224, 224), scale=(0.08, 1.0), ratio=(0.75, 1.3333), interpolation=bilinear, antialias=True)
    RandomHorizontalFlip(p=0.5)
    ColorJitter(brightness=(0.5, 1.5), contrast=(0.5, 1.5), saturation=(0.5, 1.5), hue=None)
    ToTensor()
)


## 6. 目标检测，计算机视觉训练技巧

### 6.1 理论计算题

**已知数据：**
*   真实框 $A = [10, 10, 50, 50]$
*   预测框 $B = [30, 30, 70, 70]$
*(格式均为 $[x_{left}, y_{top}, x_{right}, y_{bottom}]$)*

#### 1. 计算两个框的面积
*   $\text{Area}_A = (50 - 10) \times (50 - 10) = 40 \times 40 = 1600$
*   $\text{Area}_B = (70 - 30) \times (70 - 30) = 40 \times 40 = 1600$

#### 2. 计算交集（Intersection）的坐标
*   交集左上角：$x_{left}^{inter} = \max(10, 30) = 30$, $y_{top}^{inter} = \max(10, 30) = 30$
*   交集右下角：$x_{right}^{inter} = \min(50, 70) = 50$, $y_{bottom}^{inter} = \min(50, 70) = 50$

因为右下角坐标均大于左上角，说明存在交集区域。
$$ \text{Area}_{inter} = (50 - 30) \times (50 - 30) = 20 \times 20 = 400 $$

#### 3. 计算并集（Union）面积与 IoU
$$ \text{Area}_{union} = \text{Area}_A + \text{Area}_B - \text{Area}_{inter} = 1600 + 1600 - 400 = 2800 $$
$$ \text{IoU} = \frac{\text{Area}_{inter}}{\text{Area}_{union}} = \frac{400}{2800} = \frac{1}{7} \approx 0.1429 $$

**答：** 两个边界框之间的 IoU 准确值为 **$1/7$**（约 **$0.1429$**，或 **$14.29\%$**）。

### 6.2 编程题：实现标签平滑交叉熵损失函数

In [6]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class LabelSmoothingCrossEntropy(nn.Module):
    """
    手动实现标签平滑（Label Smoothing）交叉熵损失
    """
    def __init__(self, epsilon=0.1):
        super().__init__()
        self.epsilon = epsilon

    def forward(self, logits, targets):
        # logits 形状: (batch_size, K)
        # targets 形状: (batch_size,)，包含类别整数索引
        K = logits.size(-1)
        log_probs = F.log_softmax(logits, dim=-1)
        
        with torch.no_grad():
            # 创建平滑后的独热分布：其余错误类别设为 epsilon / (K - 1)
            smoothed_labels = torch.full_like(logits, self.epsilon / (K - 1))
            # 真实类别设为 1 - epsilon
            smoothed_labels.scatter_(1, targets.unsqueeze(1), 1.0 - self.epsilon)
            
        # 交叉熵公式: -E [p_smoothed * log(p_pred)]
        loss = (-smoothed_labels * log_probs).sum(dim=-1).mean()
        return loss

# 验证测试 (以 3 分类为例)
loss_fn = LabelSmoothingCrossEntropy(epsilon=0.1)
logits = torch.tensor([[2.0, 1.0, 0.1]], dtype=torch.float32)
targets = torch.tensor([0]) # 真实标签为第一个类别
print("标签平滑交叉熵损失输出:", loss_fn(logits, targets).item())

标签平滑交叉熵损失输出: 0.5620299577713013
